# scigantic-alphafold quickstart

Read single AlphaFold DB v6 structures with one HTTP range request each, straight from EBI's public bulk tars.

The library can reach only the v6 bulk set: Swiss-Prot (550,122 structures) and 46 proteome tars. Coordinates only, with no PAE and no MSAs. Anything else raises `NotInBulkSetError`.

The first call downloads the offset index to `~/.cache/scigantic-alphafold` (or `$SCIGANTIC_ALPHAFOLD_CACHE`). Later calls reuse it.

In [ ]:
%pip install -q scigantic-alphafold

In [ ]:
import scigantic_alphafold as af

af.index_info()

## One structure

Hemoglobin subunit alpha, UniProt accession P69905. `fetch` returns the decompressed mmCIF text.

In [ ]:
cif = af.fetch("P69905")

print(len(cif), "characters")
print("\n".join(cif.splitlines()[:12]))

`locate` shows where those bytes live: which tar, the byte offset and the length of the gzipped member. That one range is all `fetch` reads from EBI.

In [ ]:
for member in af.locate("P69905"):
    print(member)

## Per-residue pLDDT

AlphaFold stores the per-residue confidence score (pLDDT, 0 to 100) in the B-factor column of the coordinates. In mmCIF that is `_atom_site.B_iso_or_equiv`. Every atom of a residue carries the same value, so the C-alpha atoms give one value per residue.

The parser below uses only the standard library. It splits rows on whitespace, which is enough for the `_atom_site` table of AlphaFold DB protein models. For general mmCIF use a real parser such as gemmi or Biopython.

In [ ]:
def plddt_per_residue(cif_text):
    """Return [(residue number, residue name, pLDDT), ...] from the C-alpha atoms."""
    lines = cif_text.splitlines()
    start = lines.index("_atom_site.group_PDB")
    columns = []
    row = start
    while lines[row].startswith("_atom_site."):
        columns.append(lines[row].strip().split(".", 1)[1])
        row += 1
    atom = columns.index("label_atom_id")
    name = columns.index("label_comp_id")
    number = columns.index("label_seq_id")
    bfactor = columns.index("B_iso_or_equiv")
    out = []
    while row < len(lines) and lines[row].startswith(("ATOM", "HETATM")):
        fields = lines[row].split()
        if fields[atom] == "CA":
            out.append((int(fields[number]), fields[name], float(fields[bfactor])))
        row += 1
    return out


plddt = plddt_per_residue(cif)
scores = [score for _, _, score in plddt]

print(len(plddt), "residues")
print("mean pLDDT:", round(sum(scores) / len(scores), 2))
print("residues below 70:", [(number, name, score) for number, name, score in plddt if score < 70])

In [ ]:
# A text sketch of the profile: one row per 10 residues, one '#' per 5 pLDDT points.
for i in range(0, len(scores), 10):
    window = scores[i : i + 10]
    mean = sum(window) / len(window)
    print(f"{plddt[i][0]:>4}  {mean:5.1f}  {'#' * round(mean / 5)}")

## Save a file

`download` writes the structure to a directory and returns the path. The PDB format is in the bulk set too.

In [ ]:
path = af.download("P69905", "structures", fmt="pdb")
print(path, path.stat().st_size, "bytes")

## A handful of E. coli entries

`entries` iterates over the entry ids of one proteome tar without touching EBI: it reads the local index. `fetch_many` then fetches them on a small thread pool. Keep the worker count low. For a whole proteome, download EBI's tar instead: `af.proteomes()` lists the URLs.

In [ ]:
from itertools import islice

ecoli = list(islice(af.entries(proteome="UP000000625"), 5))
ecoli

In [ ]:
structures = af.fetch_many(ecoli, workers=4)

for entry_id, text in structures.items():
    values = [score for _, _, score in plddt_per_residue(text)]
    print(f"{entry_id:<16} {len(values):>5} residues  mean pLDDT {sum(values) / len(values):5.1f}")

## What is not in the bulk set

`has` answers from the index. An accession outside Swiss-Prot and the 46 proteomes raises `NotInBulkSetError`.

In [ ]:
print(af.has("P69905"), af.has("A0A000A0A0"))

try:
    af.fetch("A0A000A0A0")
except af.NotInBulkSetError as error:
    print(error)

## Metadata from EBI's prediction API

`metadata` is a passthrough of `https://alphafold.ebi.ac.uk/api/prediction/<accession>`. It needs no index and covers all of AlphaFold DB, not only the bulk set. It returns an empty list for an accession with no prediction.

In [ ]:
record = af.metadata("P69905")[0]

for key in ("entryId", "uniprotId", "uniprotDescription", "gene", "organismScientificName",
            "sequenceStart", "sequenceEnd", "globalMetricValue", "latestVersion", "modelCreatedDate"):
    print(f"{key:<24} {record[key]}")

## The bulk tars

`proteomes` is a passthrough of EBI's `download_metadata.json`, one row per tar, with the tar URL and the taxonomy id added.

In [ ]:
rows = af.proteomes()

print(len(rows), "tars,", round(sum(row["size_bytes"] for row in rows) / 1e9, 1), "GB")
for row in sorted(rows, key=lambda row: row["size_bytes"])[:5]:
    print(row["archive_name"], row["tax_id"], row["num_predicted_structures"], row["size_bytes"])

## License and citation

AlphaFold DB data is available under CC BY 4.0. If you use these structures, cite the AlphaFold DB and AlphaFold papers listed in the project README and at https://alphafold.ebi.ac.uk. This project is not affiliated with EMBL-EBI or Google DeepMind.